# Exercício 01 – Busca semântica nas reviews da Olist

**Objetivo:** entender na prática a primeira metade de um RAG, a parte de *buscar* as reviews mais parecidas com uma pergunta.

Passos:
1. Carregar as reviews
2. Transformar cada review em um vetor (embedding)
3. Fazer uma pergunta e achar as reviews mais parecidas

> Antes de começar: no Colab, clique no ícone de pasta (à esquerda) e faça upload do arquivo `olist_order_reviews_dataset.csv`.

## 1. Instalar a biblioteca de embeddings

In [ ]:
!pip install -q sentence-transformers

## 2. Carregar as reviews
Vamos pegar só as reviews que têm texto e usar uma amostra de 500 para rodar rápido.

In [ ]:
import pandas as pd

df = pd.read_csv("olist_order_reviews_dataset.csv")
print("Total de reviews:", len(df))

# fica só com as reviews que têm comentário escrito
df = df.dropna(subset=["review_comment_message"])
print("Reviews com texto:", len(df))

# amostra de 500 (random_state fixo = sempre a mesma amostra)
amostra = df.sample(500, random_state=42).reset_index(drop=True)
textos = amostra["review_comment_message"].tolist()

amostra[["review_score", "review_comment_message"]].head()

## 3. Gerar os embeddings
O modelo abaixo é multilíngue (entende português). Cada review vira uma lista de 384 números que representa o *significado* do texto.

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# normalize_embeddings=True deixa os vetores com tamanho 1,
# assim a similaridade de cosseno vira uma simples multiplicação
embeddings = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=True)

print("Formato:", embeddings.shape)   # (500, 384) = 500 reviews, 384 números cada

## 4. Função de busca
Transforma a pergunta em vetor, compara com todas as reviews e devolve as mais parecidas.

In [ ]:
import numpy as np

def buscar(pergunta, k=5):
    vetor_pergunta = modelo.encode([pergunta], normalize_embeddings=True)[0]
    similaridades = embeddings @ vetor_pergunta          # similaridade de cosseno
    melhores = np.argsort(similaridades)[::-1][:k]       # índices das k maiores
    resultado = amostra.loc[melhores, ["review_score", "review_comment_message"]].copy()
    resultado.insert(0, "similaridade", similaridades[melhores].round(3))
    return resultado

pd.set_option("display.max_colwidth", 200)
buscar("reclamação sobre atraso na entrega")

## 5. Teste com outras perguntas
Repare que as reviews que voltam nem sempre usam as mesmas palavras da pergunta. É isso que diferencia a busca semântica de um `LIKE '%atraso%'` no SQL.

In [ ]:
buscar("produto chegou quebrado ou com defeito")

In [ ]:
buscar("cliente muito satisfeito, recomenda a loja")

In [ ]:
buscar("recebi um produto diferente do que comprei")

## 6. Para pensar (e anotar)
- As reviews que voltaram fazem sentido? Alguma veio errada?
- A nota (`review_score`) das reclamações bate com o que você esperava?
- O que aconteceria se usássemos as ~40 mil reviews em vez de 500?

**Próximo passo (RAG completo):** pegar essas 5 reviews encontradas e mandar junto com a pergunta para um LLM gerar uma resposta. Isso vem nas aulas de LangChain.